In [ ]:
import pytest
from solution import Optimizer, Query

CATALOG_SIMPLE = {
    'users': {'rows': 1000},
    'orders': {'rows': 5000}
}

def make_opt(catalog, cost_model=None):
    # Support both signatures: (catalog) and (catalog, cost_model)
    try:
        if cost_model is None:
            return Optimizer(catalog)
        else:
            return Optimizer(catalog, cost_model)
    except TypeError:
        # Fallback to the other signature
        if cost_model is None:
            return Optimizer(catalog, {})
        else:
            return Optimizer(catalog)

def test_p1_optimizer_and_query_exist():
    optimizer = make_opt(CATALOG_SIMPLE)
    assert optimizer is not None
    q = Query()
    assert q is not None

def test_p1_deterministic_canonical_string():
    q1 = Query().scan('users').filter("age > 30").project(['id', 'name'])
    q2 = Query().scan('users').filter("age > 30").project(['id', 'name'])

    opt = make_opt(CATALOG_SIMPLE)
    plan1 = opt.get_plan(q1)
    plan2 = opt.get_plan(q2)

    assert isinstance(plan1, str) and plan1
    assert plan1 == plan2

def test_p1_different_queries_produce_different_strings():
    opt = make_opt(CATALOG_SIMPLE)

    q1 = Query().scan('users').project(['id', 'name'])
    q2 = Query().scan('users').filter("age > 30")
    q3 = Query().scan('orders')

    p1 = opt.get_plan(q1)
    p2 = opt.get_plan(q2)
    p3 = opt.get_plan(q3)

    assert p1 != p2
    assert p1 != p3
    assert p2 != p3

In [ ]:
import pytest
from solution import Optimizer, Query

CATALOG_SIMPLE = {'users': {'rows': 1000}}

def make_opt(catalog, cost_model=None):
    try:
        if cost_model is None:
            return Optimizer(catalog)
        else:
            return Optimizer(catalog, cost_model)
    except TypeError:
        if cost_model is None:
            return Optimizer(catalog, {})
        else:
            return Optimizer(catalog)

def test_p2_add_rule_exists_and_callable():
    opt = make_opt(CATALOG_SIMPLE)
    opt.add_rule(lambda node: node)  # no-op rule

def test_p2_rules_are_invoked_but_not_required_to_change_plan():
    opt = make_opt(CATALOG_SIMPLE)
    q = Query().scan('users')

    calls = {'count': 0}
    def counting_rule(node):
        calls['count'] += 1
        return node  # no change required

    opt.add_rule(counting_rule)
    plan = opt.get_plan(q)
    assert isinstance(plan, str) and plan
    assert calls['count'] > 0  # rule was invoked at least once

def test_p2_determinism_with_rules():
    opt = make_opt(CATALOG_SIMPLE)
    q = Query().scan('users')

    opt.add_rule(lambda node: node)
    opt.add_rule(lambda node: node)

    p1 = opt.get_plan(q)
    p2 = opt.get_plan(q)
    assert p1 == p2

In [ ]:
import pytest
from solution import Optimizer, Query

CATALOG = {
    'users': {'rows': 1000}
}

def make_opt(catalog, cost_model=None):
    try:
        if cost_model is None:
            return Optimizer(catalog)
        else:
            return Optimizer(catalog, cost_model)
    except TypeError:
        if cost_model is None:
            return Optimizer(catalog, {})
        else:
            return Optimizer(catalog)

def test_p3_required_order_is_accepted_and_reflected():
    """
    If a required order is provided, the plan string must clearly reflect it.
    We do NOT require a specific operator name, only that the required
    column shows up in the final plan string.
    """
    opt = make_opt(CATALOG)
    q = Query().scan('users').project(['id', 'age', 'name'])

    plan = opt.get_plan(q, required_order=['age'])
    assert isinstance(plan, str) and plan
    assert 'age' in plan

def test_p3_multiple_order_columns_are_reflected_in_sequence():
    opt = make_opt(CATALOG)
    q = Query().scan('users').project(['id', 'age', 'name'])

    plan = opt.get_plan(q, required_order=['age', 'name'])
    assert isinstance(plan, str) and plan
    assert 'age' in plan and 'name' in plan
    # Robust check: ensure there exists an occurrence of 'age' before some
    # later occurrence of 'name' (tolerates schema lists earlier in the string).
    assert plan.find('age') < plan.rfind('name')

def test_p3_nonexistent_order_column_raises():
    """
    Error when ordering by a non-existent column, relative to the query's output.
    """
    opt = make_opt(CATALOG)
    q = Query().scan('users').project(['id'])  # 'zzz' not part of output

    with pytest.raises(Exception):
        _ = opt.get_plan(q, required_order=['zzz'])

In [ ]:
import pytest
from solution import Optimizer, Query

CATALOG = {'users': {'rows': 1000, 'columns': ['id', 'name', 'age']}}

def test_p4_cost_model_is_accepted_and_plan_is_deterministic():
    """
    We only assert that a cost model can be provided and that determinism holds.
    Implementations are free to design their own exploration space and costing.
    """
    cost_model = {'scan': 1.0, 'filter': 0.5, 'project': 0.25, 'sort': 10.0}
    opt = Optimizer(CATALOG, cost_model)
    q = Query().scan('users').project(['name'])

    p1 = opt.get_plan(q)
    p2 = opt.get_plan(q)
    assert isinstance(p1, str) and p1
    assert p1 == p2

def test_p4_required_order_is_enforced_even_if_sort_is_costly():
    """
    Cost should not let the optimizer ignore required order. We do not
    require a specific operator name; just ensure required columns appear.
    """
    cost_model = {'scan': 1.0, 'sort': 1e9}
    opt = Optimizer(CATALOG, cost_model)
    q = Query().scan('users')

    plan = opt.get_plan(q, required_order=['name'])
    assert 'name' in plan

In [ ]:
import pytest
from solution import Optimizer, Query

CATALOG = {
    'a': {'rows': 100},
    'b': {'rows': 10000},
    'c': {'rows': 50}
}
COST_MODEL = {'scan': 1.0, 'join': 5.0}

def test_p5_join_reordering_produces_stable_optimal_plan():
    """
    The optimizer should produce the same final plan regardless of how
    the initial query is associated. We do not enforce a particular order,
    only stability across equivalent inputs.
    """
    opt = Optimizer(CATALOG, COST_MODEL)

    # Left-deep association
    q_left = Query().scan('a').join(
        Query().scan('b'), "a.id=b.a_id"
    ).join(
        Query().scan('c'), "b.id=c.b_id"
    )
    # Right-deep association (if Query allows it via nested joins)
    q_right = Query().scan('c').join(
        Query().scan('b'), "b.id=c.b_id"
    ).join(
        Query().scan('a'), "a.id=b.a_id"
    )

    p_left = opt.get_plan(q_left)
    p_right = opt.get_plan(q_right)
    assert isinstance(p_left, str) and p_left
    assert p_left == p_right  # Optimal plan should be canonical/stable

In [ ]:
import pytest
from solution import Optimizer, Query

CATALOG = {
    'users': {'rows': 1000, 'columns': ['id', 'country']},
    'orders': {'rows': 5000, 'columns': ['id', 'user_id', 'amount']}
}
COST_MODEL = {'scan': 1.0, 'filter': 0.1, 'join': 10.0}

def test_p6_late_filter_and_early_filter_result_in_same_plan():
    """
    A filter applied after a join should be pushed down to earlier operators.
    We test this by comparing with a query built with early filters.
    """
    late = Query().scan('users').join(
        Query().scan('orders'), "users.id=orders.user_id"
    ).filter("users.country = 'US'")

    early = Query().scan('users').filter("country = 'US'").join(
        Query().scan('orders'), "users.id=orders.user_id"
    )

    opt = Optimizer(CATALOG, COST_MODEL)
    p_late = opt.get_plan(late)
    p_early = opt.get_plan(early)
    assert p_late == p_early

def test_p6_split_compound_predicate_across_join_is_equivalent():
    """
    A combined predicate referencing both tables should be split and pushed
    to inputs where possible. We compare the late form with an explicitly
    early-split form.
    """
    late = Query().scan('users').join(
        Query().scan('orders'), "users.id=orders.user_id"
    ).filter("users.country = 'US' and orders.amount > 100")

    split_early = Query().scan('users').filter("country = 'US'").join(
        Query().scan('orders').filter("amount > 100"),
        "users.id=orders.user_id"
    )

    opt = Optimizer(CATALOG, COST_MODEL)
    p1 = opt.get_plan(late)
    p2 = opt.get_plan(split_early)
    assert p1 == p2

In [ ]:
import pytest
from solution import Optimizer, Query

# Enhanced stats including distinct values per column
CATALOG = {
    'users': {
        'rows': 10000,
        'columns': {
            'id': {'distinct_values': 10000},
            'status': {'distinct_values': 5},
            'email': {'distinct_values': 10000}
        }
    },
    'audits': {
        'rows': 1000,
        'columns': {
            'user_id': {'distinct_values': 500}
        }
    }
}
COST = {'scan': 0.1, 'filter': 0.1, 'join': 0.2}

def test_p7_optimizer_accepts_enhanced_statistics_and_is_deterministic():
    """
    Minimal check: the optimizer consumes per-column stats and remains deterministic.
    The exact estimation formulas and plan choices are left to the implementation.
    """
    opt = Optimizer(CATALOG, COST)
    q = Query().scan('users').filter("status = 'active'").join(
        Query().scan('audits'), "users.id=audits.user_id"
    )

    p1 = opt.get_plan(q)
    p2 = opt.get_plan(q)
    assert isinstance(p1, str) and p1
    assert p1 == p2

In [ ]:
import re
import pytest
from solution import Optimizer, Query

CATALOG = {
    'users': {'rows': 1000, 'columns': {'id': {}, 'status': {}}},
    'banned': {'rows': 50, 'columns': {'user_id': {}}},
    'orders': {'rows': 5000, 'columns': {'user_id': {}}},
}
COST = {'scan': 1.0, 'join': 5.0, 'filter': 1.0, 'semi_join': 4.0}


def _call_any(obj, names, *args):
    """
    Try calling any of the named methods on obj with args.
    Return (result, name) or (None, None) if not found.
    """
    for name in names:
        fn = getattr(obj, name, None)
        if callable(fn):
            try:
                return fn(*args), name
            except TypeError:
                # Signature mismatch; try next
                continue
    return None, None


def _assert_no_explicit_subquery_tokens(plan: str):
    """
    Ensure the plan string does not contain explicit IN/EXISTS/subquery
    tokens. Robust to whitespace/case; avoids matching 'join(' by requiring
    word boundaries.
    """
    assert re.search(r'\bin\s*\(', plan, flags=re.IGNORECASE) is None
    assert re.search(r'\bexists\s*\(', plan, flags=re.IGNORECASE) is None
    assert re.search(r'\bsubquery\b', plan, flags=re.IGNORECASE) is None


def _is_join_like(plan: str) -> bool:
    s = plan.lower()
    return ('join' in s) or ('semi' in s)


def test_p8_in_subquery_is_rewritten_to_join_like_plan():
    """
    Accepts multiple possible APIs to express IN-subqueries.
    Fallback: a string-based filter that includes 'IN (...)' so
    parser-based solutions can pass. The final plan must not contain
    explicit subquery artifacts ('IN(', 'EXISTS(', 'subquery').
    It must show a join-like plan ('join' or 'semi').
    """
    sub = Query().scan('banned').project(['user_id'])
    base = Query().scan('users')

    # Try a variety of likely method names
    q, used = _call_any(
        base,
        ['filter_in', 'where_in', 'in_', 'in_subquery', 'add_in_filter'],
        'id',
        sub,
    )
    if q is None:
        # Very permissive fallback: allow a string-based IN filter for parsers
        # Models that didn't implement a special API can still pass if they
        # decorrelate strings.
        q = base.filter("id IN (SELECT user_id FROM banned)")

    opt = Optimizer(CATALOG, COST)
    plan = opt.get_plan(q)

    _assert_no_explicit_subquery_tokens(plan)
    assert _is_join_like(plan)


def test_p8_exists_subquery_is_rewritten():
    """
    Accepts multiple APIs for EXISTS. Correlation can be expressed either by
    a dedicated API or by a filter inside the subquery referencing the outer
    alias.
    """
    sub = Query().scan('orders')
    # Try to express correlation inside the subquery using a flexible approach:
    # If there is a correlation-specific API, use it; otherwise use a normal
    # filter with a string predicate.
    correlated_sub, used_corr = _call_any(
        sub,
        ['filter_by_outer', 'correlate_on', 'where_outer_eq', 'outer_eq'],
        "user_id",
        "users.id",
    )
    if correlated_sub is None:
        correlated_sub = sub.filter("orders.user_id = users.id")

    base = Query().scan('users')

    q, used = _call_any(
        base,
        [
            'filter_exists',
            'where_exists',
            'exists',
            'exists_subquery',
            'add_exists_filter',
        ],
        correlated_sub,
    )
    if q is None:
        # Fallback string-based EXISTS for parsers
        q = base.filter(
            "EXISTS (SELECT 1 FROM orders WHERE orders.user_id = users.id)"
        )

    opt = Optimizer(CATALOG, COST)
    plan = opt.get_plan(q)

    _assert_no_explicit_subquery_tokens(plan)
    assert _is_join_like(plan)

In [ ]:
import re
import pytest
from solution import Optimizer, Query

CATALOG = {
    'users': {'rows': 1000, 'columns': {'id': {}, 'name': {}}},
    'logs': {'rows': 100000, 'columns': {'user_id': {}}},
}

# Cost keys name the physical algorithms; the plan should also reflect one.
COST = {
    'scan': 1.0,
    'sort': 10.0,
    'HashJoin': 20.0,
    'SortMergeJoin': 5.0,
    'NestedLoopJoin': 100.0,
}


def _algo_keys(cost):
    # Accept any cost key that looks like a join algorithm (but not a generic
    # 'join')
    return [k for k in cost.keys() if 'join' in k.lower() and k.lower() != 'join']


def _norm_token(s: str) -> str:
    # Normalize tokens for matching regardless of case/spacing/underscores/hyphens
    return re.sub(r'[^a-z0-9]', '', s.lower())


def _contains_required_order_token(plan: str, required_col: str) -> bool:
    """
    Check that the required column is reflected in the plan. Prefer an exact
    token match; otherwise accept an unqualified column appearing in an
    obvious sort/order context to tolerate implementations that dequalify.
    """
    if re.search(re.escape(required_col), plan, flags=re.IGNORECASE):
        return True
    unqual = required_col.split('.')[-1]
    if re.search(r'(order|sort)[^a-z0-9]{0,40}.*' + re.escape(unqual),
                 plan,
                 flags=re.IGNORECASE):
        return True
    return False


def test_p9_plan_names_a_physical_join_algorithm():
    q = Query().scan('users').join(Query().scan('logs'), "users.id=logs.user_id")
    opt = Optimizer(CATALOG, COST)
    plan = opt.get_plan(q)

    algos = _algo_keys(COST)
    plan_n = _norm_token(plan)
    algos_n = [_norm_token(a) for a in algos]

    # At least one algorithm name (normalized) should appear in the plan
    assert any(a in plan_n for a in algos_n), (
        f"Plan must name a physical join algorithm from {algos}"
    )


def test_p9_required_order_can_influence_physical_choice():
    """
    Requiring a final order may influence the physical algorithm choice.
    We don't force a particular algorithm (like SortMergeJoin); we only assert
    determinism and that the plan can differ when an order is required.
    """
    q = Query().scan('users').join(Query().scan('logs'), "users.id=logs.user_id")
    opt = Optimizer(CATALOG, COST)

    p_no_order_1 = opt.get_plan(q)
    p_no_order_2 = opt.get_plan(q)
    assert p_no_order_1 == p_no_order_2  # determinism

    p_with_order = opt.get_plan(q, required_order=['users.id'])
    # Either the plan changes (influenced by required order) or stays the same.
    # Both are acceptable, but if it changes, it must remain deterministic:
    p_with_order_2 = opt.get_plan(q, required_order=['users.id'])
    assert p_with_order == p_with_order_2

    # If an order is required, it should be reflected in the plan string
    assert _contains_required_order_token(p_with_order, 'users.id')

In [ ]:
import pytest
from solution import Optimizer, Query

CATALOG = {'a': {'rows': 100}, 'b': {'rows': 100}}
COST = {'scan': 1, 'HashJoin': 10, 'NestedLoopJoin': 100, 'SortMergeJoin': 50}

def test_p10_hints_parameter_is_accepted_and_deterministic():
    """
    We only require that hints are accepted and planning remains deterministic.
    Specific hint syntax and effects are left to the implementation.
    """
    q = Query().scan('a').join(Query().scan('b'), "a.id=b.id")
    opt = Optimizer(CATALOG, COST)
    p1 = opt.get_plan(q, hints=['SOME_HINT'])
    p2 = opt.get_plan(q, hints=['SOME_HINT'])
    assert isinstance(p1, str) and p1
    assert p1 == p2

def test_p10_impossible_hint_may_raise_error_but_not_required():
    """
    If the implementation detects impossible hints, it can raise an Exception.
    If it silently ignores unknown hints, that's acceptable too.
    """
    q = Query().scan('a').join(Query().scan('b'), "a.id=b.id")
    opt = Optimizer(CATALOG, COST)
    try:
        _ = opt.get_plan(q, hints=['FORCE_NONEXISTENT_ALGO'])
    except Exception:
        # Acceptable behavior: explicit error on impossible hint
        pass



CATALOG = {'a': {'rows': 100}, 'b': {'rows': 100}}
COST = {'scan': 1, 'HashJoin': 10, 'NestedLoopJoin': 100, 'SortMergeJoin': 50}

def test_p10_set_plan_stability_exists_and_is_harmless():
    opt = Optimizer(CATALOG, COST)
    assert hasattr(opt, 'set_plan_stability'), "Optimizer must expose set_plan_stability"
    # Call it; we do not prescribe semantics here
    try:
        opt.set_plan_stability(tolerance_percent=25.0)
    except TypeError:
        # Allow alternative signatures (e.g., enabled flag); prompts kept minimal
        opt.set_plan_stability(25.0)

    q = Query().scan('a').join(Query().scan('b'), "a.id=b.id")
    p1 = opt.get_plan(q)
    p2 = opt.get_plan(q)
    assert p1 == p2  # determinism remains intact